In [ ]:
from deepcave import Recorder, Objective
from deepcave.runs.converters.deepcave import DeepCAVERun
from deepcave.plugins.hyperparameter.importances import Importances
import pandas as pd
from arlbench.core.algorithms import DQN, PPO, SAC
from pathlib import Path
from ConfigSpace import ConfigurationSpace, Float, Categorical
import math
import os

# Bei mir gabs Fehler mit bool_ logging, ich musste in deepcave/runs/run.py folgendes einfügen (nach Zeile 355):
# for conf in self.configs:
#     for k in self.configs[conf].keys():
#         if isinstance(self.configs[conf][k], np.bool_):
#             self.configs[conf][k] = bool(self.configs[conf][k])

In [2]:
replacement_dict = {"atari_qbert": "Atari", 'atari_double_dunk': "Atari", 'atari_phoenix': "Atari", 'atari_this_game': "Atari", 'atari_battle_zone': "Atari", 
                    'box2d_lunar_lander': "Box2d", "box2d_bipedal_walker": "Box2d", 'cc_acrobot': "CC", 'cc_cartpole': "CC", 'cc_mountain_car': "CC", "cc_pendulum": "CC", 'cc_continuous_mountain_car': "CC",
                    'minigrid_door_key': "XLand", 'minigrid_empty_random': "XLand", 'minigrid_four_rooms': "XLand", 
                    'minigrid_unlock': "XLand", "brax_ant": "Brax", "brax_halfcheetah": "Brax", "brax_hopper": "Brax", "brax_humanoid": "Brax"}
algorithms = {"dqn": DQN, "ppo": PPO, "sac": SAC}
envs = ["atari_qbert", 'atari_double_dunk', 'atari_phoenix', 'atari_this_game', 'atari_battle_zone', 
        'box2d_lunar_lander', 'box2d_bipedal_walker', 'cc_acrobot', 'cc_cartpole', 'cc_mountain_car', 'cc_pendulum', 'cc_continuous_mountain_car',
        'minigrid_door_key', 'minigrid_empty_random', 'minigrid_four_rooms', 'minigrid_unlock', 'brax_ant', 'brax_halfcheetah', 'brax_hopper', 'brax_humanoid']

In [3]:
def data_to_deepcave(algorithm, domain=False, save_path=None):
    target_path = f"arlbench_data/256_10/{domain}_{algorithm}.csv"
    try:
        all_configs = pd.read_csv(target_path)
    except:
        print(f"Could not read {target_path}")
        return

    default_configspace = algorithms[algorithm].get_hpo_search_space().get_hyperparameter_names()
    configspace = ConfigurationSpace()
    for c in default_configspace:
        if c in ["buffer_prio_sampling", "use_target_network", "alpha_auto", "normalize_advantage"]:
            hp = Categorical(c, [True, False])
            configspace.add_hyperparameter(hp)
        else:
            key = [a for a in all_configs.keys() if c in a]
            if len(key) > 0:
                key = key[0]
                hp_min = all_configs[key].min()
                hp_max = all_configs[key].max()
                bounds = (min(0, hp_min-0.1*hp_min), hp_max+0.1*hp_max)
                hp = Float(c, bounds=bounds)
                configspace.add_hyperparameter(hp)
    default = configspace.get_default_configuration()
    accuracy = Objective("reward_mean", lower=min(all_configs["last_performance"]), upper=max(all_configs["last_performance"]), optimize="upper")
    save_path = save_path if save_path else f"deepcave_logs/{algorithm}_{domain}"

    with Recorder(configspace, objectives=[accuracy], save_path=save_path) as r:
        for index, d in all_configs.iterrows():
            current_hps = d
            config = default
            for c in current_hps.keys():
                key = c.split(".")[-1]
                if key in config.keys():
                    if math.isnan(current_hps[c]):
                        current_hps[c] = 0
                    try:
                        value = float(current_hps[c])
                    except:
                        current_hps[c]
                    config[key] = value
        
            r.start(config=config, budget=1)
            r.end(costs=current_hps["last_performance"], budget=1)
    return save_path

In [4]:
def get_importances(path, algorithm, method="local"):
    # Instantiate the run
    run = DeepCAVERun.from_path(Path(path))
    objective_id = run.get_objective_ids()[0]
    budget_ids = run.get_budget_ids()

    # Instantiate the plugin
    plugin = Importances()

    inputs = plugin.generate_inputs(
        objective_id=objective_id,
        hyperparameter_names= algorithms[algorithm].get_hpo_search_space().get_hyperparameter_names(), 
        budget_ids=budget_ids,
        method=method, n_hps=10, n_trees=10
    )
    processed = plugin.process(run, inputs)

    # Note: Filter variables are not considered.
    outputs = plugin.generate_outputs(run, inputs)

    # Finally, you can load the figure. Here, the filter variables play a role.
    # Alternatively: Use the matplotlib output (`load_mpl_outputs`) if available.
    figure = plugin.load_outputs(run, inputs, outputs)  # plotly.go figure
    return figure, inputs, outputs, processed


def get_importances_values(path, algorithm, method="local", threshold=0.05):
    from deepcave.evaluators.fanova import fANOVA
    
    # Instantiate the run
    run = DeepCAVERun.from_path(Path(path))

    # Instantiate the plugin
    fanova = fANOVA(run=run)

    fanova.calculate(n_trees=10)

    importances = fanova.get_importances()

    importances_over_threshold = {}

    for hp, importance in importances.items():
        importance = importance[0]  # we use the mean

        if importance >= threshold:
            importances_over_threshold[hp] = importance    

    return importances_over_threshold

In [5]:
hps = {"algos": [], "envs": [], "hp_name": [], "hp_importance": []}
for algorithm in ["ppo", "dqn", "sac"]:
    deepcave_paths = []
    for env in envs:
        if not os.path.isdir(f"deepcave_logs/{algorithm}_{env}"):
            print(f"{env} not found")

    for env in envs:
        if not os.path.isdir(f"deepcave_logs/{algorithm}_{env}"):
            data_path = data_to_deepcave(algorithm, domain=env)
        deepcave_paths.append(f"deepcave_logs/{algorithm}_{env}")
    
    for deepcave_path in deepcave_paths:
        if deepcave_path is None:
            continue
        try:
            _, _, outs, _ = get_importances(Path(deepcave_path)/"run_1", algorithm=algorithm, method="global")
            importances = outs[0]
            for hp in importances.keys():
                hps["hp_name"].append(hp)
                hps["hp_importance"].append(importances[hp][0])
                hps["algos"].append(algorithm)
                hps["envs"].append("_".join(deepcave_path.split("_")[2:]))
        except:
            continue
importances_df = pd.DataFrame(hps)

/var/folders/yc/_2wjgdhd4px91396bq5wm7kr0000gn/T/ipykernel_94741/2480975025.py:12: DeprecationWarning: Please use `list(space.keys())`
  hyperparameter_names= algorithms[algorithm].get_hpo_search_space().get_hyperparameter_names(),


0.185260766279649 1.0
0.0616986771062325 1.0
0.0062113705060799 1.0
0.0468649438565335 1.0
0.0209799767514286 1.0
0.1962060085895277 1.0
0.0750702355623688 1.0
0.0114509373781917 1.0
0.0214690711147786 1.0
0.9696927552771282 1.0
0.0137916943802762 1.0
0.013487766333106 1.0
0.0075483987374519 1.0
0.0210527659062868 1.0
0.0560657917386837 1.0
0.0483947998353816 1.0
0.006374827624661 1.0
0.0102837514888982 1.0
0.0070286560989852 1.0
0.0055575420317557 1.0
0.044903458368178 1.0
0.0469428411117301 1.0
0.0347984881681107 1.0
0.0060479133874989 1.0
0.0084997701662147 1.0
0.0026153138972968 1.0
0.2389245045967803 1.0
0.0887201815639828 1.0
0.0156995453381402 1.0
0.1117510347097137 1.0
0.0 1.0
0.1971318374897297 1.0
0.0407212543395903 1.0
0.0053940849131747 1.0
0.0158732185920154 1.0
0.0088266844033768 1.0
0.0080477091053304 1.0
0.05126040731474 1.0
0.0221880268549846 1.0
0.2688154488841083 1.0
0.2039574524520241 1.0
0.0143471932307411 1.0
0.1961319413307679 1.0
0.0152091740477799 1.0
0.0408642

/var/folders/yc/_2wjgdhd4px91396bq5wm7kr0000gn/T/ipykernel_94741/2480975025.py:12: DeprecationWarning:

Please use `list(space.keys())`



0.0173340748980346 1.0
0.0123285131627734 1.0
0.0235446792732664 1.0
0.0151093807934741 1.0
0.0108453837597328 1.0
0.1529477196885427 1.0
0.2157953281423804 1.0
0.0245643307378568 1.0
0.0258620689655171 1.0
0.4605580274378938 1.0
0.0078791249536522 1.0
0.0168705969595845 1.0
0.0174267704857248 1.0
0.0131627734519835 1.0
0.1142936596218019 1.0
0.0022246941045604 1.0
0.0081572117167222 1.0
0.0174267704857248 1.0
0.0065813867259917 1.0
0.0116796440489431 1.0
0.0240081572117165 1.0
0.0604375231738967 1.0
0.011216166110493 1.0
0.0217834631071561 1.0
0.0224323322209861 1.0
0.0230812013348163 1.0
0.0281794586577677 1.0
0.0246570263255468 1.0
0.0179829440118649 1.0
0.1051167964404893 1.0
0.0200222469410455 1.0
0.0253985910270669 1.0
0.0186318131256951 1.0
0.0177975528364848 1.0
0.0448646644419724 1.0
0.0082499073044121 1.0
0.0170559881349646 1.0
0.0569150908416758 1.0
0.0329069336299592 1.0
0.0231738969225064 1.0
0.0192806822395253 1.0
0.0153874675565442 1.0
0.0407860585836114 1.0
0.0067667779

/var/folders/yc/_2wjgdhd4px91396bq5wm7kr0000gn/T/ipykernel_94741/2480975025.py:12: DeprecationWarning:

Please use `list(space.keys())`



...

0.2196986381976105 1.0
0.0367882932007447 1.0
0.5428175696889683 1.0
0.5384388637667714 1.0
0.3009084675537183 1.0
0.7743755981402183 1.0
0.2072694287354812 1.0
0.4215820954658755 1.0
0.5553278991308425 1.0
0.5854720417976286 1.0
0.590588289114844 1.0
0.3818695718133572 1.0
0.7272918300902783 1.0
0.175965809498519 1.0
0.975495990288429 1.0
0.3963791885450106 1.0
0.0334439083121749 1.0
0.8030882316915637 1.0
0.81322596787962 1.0
0.9281001009034752 1.0
0.2644892566506238 1.0
0.3521593515086963 1.0
0.036453015131852 1.0
0.0503975390678159 1.0
0.9292385911867874 1.0
0.8916843664758906 1.0
0.8499251246992532 1.0
0.0369593836646448 1.0
0.3118055946282755 1.0
0.0175278420325826 1.0
0.4070674113691817 1.0
0.4026297798931066 1.0
0.8212981665858327 1.0
0.5048959160781274 1.0
0.0369386479543177 1.0
0.1163437935611388 1.0
0.1104720768254941 1.0
0.0328488775639324 1.0
0.0369648730505248 1.0
0.2083821346967101 1.0
0.0355810650836926 1.0
0.0369612882122937 1.0
0.3254306026216561 1.0
0.826326322859864

/var/folders/yc/_2wjgdhd4px91396bq5wm7kr0000gn/T/ipykernel_94741/2480975025.py:12: DeprecationWarning:

Please use `list(space.keys())`



0.514180869089475 1.0
0.3005417908503753 1.0
0.3778795909912778 1.0
0.5993799713940124 1.0
0.8996826350816891 1.0
0.7414181706995426 1.0
0.5581035098755188 1.0
0.6256837740793708 1.0
0.72720393606688 1.0
0.3000593280914126 1.0
0.5940362847863316 1.0
0.8299639374209198 1.0
0.7136866323597715 1.0
0.426235766179444 1.0
0.4600431076428951 1.0
0.8427794212314956 1.0
0.253321501790903 1.0
0.4927976335438345 1.0
0.8015001340898252 1.0
0.3569655821651994 1.0
0.2010931467872586 1.0
0.7290816376707806 1.0
0.6790549497746772 1.0
0.5708330035133864 1.0
0.4855690600449749 1.0
0.4190824363011804 1.0
0.6463179675343431 1.0
0.0441746935675685 1.0
0.3973699369788421 1.0
0.617615344552845 1.0
0.7511635019649587 1.0
0.8922941343723765 1.0
0.6854073517122895 1.0
0.3663883953317833 1.0
0.0291284330366265 1.0
0.6853220792478681 1.0
0.3623638914669804 1.0
0.3021911285280299 1.0
0.0314353206571619 1.0
0.6256244953827901 1.0
0.3863088945379178 1.0
0.0282029526365915 1.0
0.6999087100694813 1.0
0.842862383732132

/var/folders/yc/_2wjgdhd4px91396bq5wm7kr0000gn/T/ipykernel_94741/2480975025.py:12: DeprecationWarning:

Please use `list(space.keys())`



0.5428639551727797 1.0
0.104692592742583 1.0
0.5521102442760595 1.0
0.6724488787397058 1.0
0.6756849626931484 1.0
0.5620984804625582 1.0
0.3734554963949526 1.0
0.6077396434655198 1.0
0.004723902531585 1.0
0.4766923327031741 1.0
0.7080355816882522 1.0
0.5967120285912526 1.0
0.5803406430910362 1.0
0.3545823572357291 1.0
0.7640248018915845 1.0
0.5179917191314912 1.0
0.056622945854014 1.0
0.7587297531976285 1.0
0.3662631261151097 1.0
0.3329128526645255 1.0
0.1483408409242579 1.0
0.566581948923469 1.0
0.5844448213520688 1.0
0.2289091785925812 1.0
0.6673374604271468 1.0
0.6991529101203251 1.0
0.395845934260829 1.0
0.0174244452383703 1.0
0.7011801120830332 1.0
0.5241879480730472 1.0
0.4533155315791864 1.0
0.0473756553514526 1.0
0.3552831677019852 1.0
0.5650554744221474 1.0
0.0176373498634517 1.0
0.0240381038664662 1.0
0.1527053090633071 1.0
0.047588609575797 1.0
0.0174677997713794 1.0
0.5763399617711694 1.0
0.0697160246583236 1.0
0.0173730244320396 1.0
0.6203715528199714 1.0
0.499612956760898

In [6]:
print(len(hps["hp_name"]))
print(len(hps["hp_importance"]))
print(len(hps["algos"]))
print(len(hps["envs"]))
print(hps.keys())

406
406
406
406
dict_keys(['algos', 'envs', 'hp_name', 'hp_importance'])


In [7]:
print(importances_df)

    algos           envs              hp_name  hp_importance
0     ppo    atari_qbert  normalize_advantage       0.000127
1     ppo    atari_qbert          vf_clip_eps       0.000241
2     ppo    atari_qbert       minibatch_size       0.001968
3     ppo    atari_qbert           gae_lambda       0.002276
4     ppo    atari_qbert             clip_eps       0.006317
..    ...            ...                  ...            ...
401   sac  brax_humanoid          buffer_size       0.000716
402   sac  brax_humanoid         buffer_alpha       0.005506
403   sac  brax_humanoid                  tau       0.009740
404   sac  brax_humanoid      learning_starts       0.011311
405   sac  brax_humanoid        learning_rate       0.640204

[406 rows x 4 columns]


In [8]:
import seaborn as sns
sns.catplot(data=importances_df, x="hp_name", y="hp_importance", col="algos", kind="box")

matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in

In [19]:
importances_df["domain"] = importances_df["envs"].apply(lambda x: x.split("_")[0])
print(importances_df["domain"])

0      atari
1      atari
2      atari
3      atari
4      atari
       ...  
401     brax
402     brax
403     brax
404     brax
405     brax
Name: domain, Length: 406, dtype: object


In [20]:
for algo in ["dqn", "ppo", "sac"]:
    cat = sns.relplot(data=importances_df[importances_df["algos"]==algo], x="hp_name", y="hp_importance", col="algos", hue="domain", kind="scatter")
    for ax in cat.axes.flat:
        for label in ax.get_xticklabels():
            label.set_rotation(45)
    cat.savefig(f"importances_{algo}_scatter")

matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in

In [13]:
importances_df.to_csv("importance_data.csv")

In [28]:
smac_data = pd.read_csv("smac_on_dataset.csv")
print(smac_data[(smac_data["budget"]==32)&(smac_data["algorithm"]=="ppo")&(smac_data["env_name"]=="atari_qbert")])

     Unnamed: 0 algorithm     env_name  budget  incumbent  mean_performance
38           38       ppo  atari_qbert      32  -0.189875         -0.061118
190         190       ppo  atari_qbert      32  -0.189875         -0.061118
342         342       ppo  atari_qbert      32  -0.189875         -0.061118
494         494       ppo  atari_qbert      32  -0.189875         -0.061118
646         646       ppo  atari_qbert      32  -0.189875         -0.061118


In [27]:
import matplotlib.pyplot as plt
plt.figure()
pp = sns.pointplot(data=smac_data, x="budget", y="incumbent", hue="algorithm")
pp.get_figure().savefig("smac_incumbents")
plt.figure()
pp = sns.pointplot(data=smac_data, x="budget", y="mean_performance", hue="algorithm")
pp.get_figure().savefig("smac_means")

matplotlib.category (INFO): Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
matplotlib.category (INFO): Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
matplotlib.texmanager (INFO): No LaTeX-compatible font found for the serif fontfamily in rcParams. Using def